In [ ]:
# [Cell 1] Install dependencies
# !pip install transformer_lens sae_lens transformers torch
# !pip install --force-reinstall numpy scipy pandas scikit-learn

# If on Colab, restart runtime after install:
# Runtime -> Restart runtime (Ctrl+M+.), then run from Cell 2

## Setup
Load GPT-2 and SAE weights.

In [1]:
# [Cell 2] Load GPT-2 model
import torch
import transformer_lens as tl
import os
# from google.colab import drive

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using: {device}')

# --- Colab (Google Drive) ---
# drive.mount('/content/drive')
# save_path = '/content/drive/MyDrive/gpt2_small_hooked.pt'

# --- Local ---
save_dir = os.path.dirname(os.path.abspath('__file__'))
save_path = os.path.join(save_dir, 'gpt2_small_hooked.pt')

if os.path.exists(save_path):
    model = tl.HookedTransformer.from_pretrained('gpt2-small', device='cpu')
    model.load_state_dict(torch.load(save_path, map_location=device))
    model.to(device)
    print("Loaded from local cache")
else:
    model = tl.HookedTransformer.from_pretrained('gpt2-small', device=device)
    torch.save(model.state_dict(), save_path)
    print(f"Downloaded and saved to {save_path}")

print(f'GPT-2 small: {sum(p.numel() for p in model.parameters())/1e6:.0f}M params')
print(f'dim={model.cfg.d_model}, layers={model.cfg.n_layers}, heads={model.cfg.n_heads}')

Using: cpu


`torch_dtype` is deprecated! Use `dtype` instead!


Loaded pretrained model gpt2-small into HookedTransformer
Moving model to device:  cpu
Loaded from local cache
GPT-2 small: 163M params
dim=768, layers=12, heads=12


In [3]:
# [Cell 3] Load SAE as raw tensors
# sae_lens downloads the weights, then we extract 4 tensors and only use those
from sae_lens import SAE

_sae = SAE.from_pretrained(
    release="gpt2-small-res-jb",
    sae_id="blocks.8.hook_resid_pre",
    device=device
)

# Extract the 4 tensors that define the entire SAE
W_enc = _sae.W_enc.detach()   # [768, 24576]  encoder weights
b_enc = _sae.b_enc.detach()   # [24576]       encoder bias
W_dec = _sae.W_dec.detach()   # [24576, 768]  decoder weights
b_dec = _sae.b_dec.detach()   # [768]         decoder bias
del _sae

# SAE encode = ReLU((x - b_dec) @ W_enc + b_enc)
# SAE decode = features @ W_dec + b_dec
print(f"W_enc: {W_enc.shape}  (768 -> 24576)")
print(f"b_enc: {b_enc.shape}")
print(f"W_dec: {W_dec.shape}  (24576 -> 768)")
print(f"b_dec: {b_dec.shape}")

W_enc: torch.Size([768, 24576])  (768 -> 24576)
b_enc: torch.Size([24576])
W_dec: torch.Size([24576, 768])  (24576 -> 768)
b_dec: torch.Size([768])


/home/cody/anaconda3/lib/python3.12/site-packages/sae_lens/saes/sae.py:251: UserWarning: 
This SAE has non-empty model_from_pretrained_kwargs. 
For optimal performance, load the model like so:
model = HookedSAETransformer.from_pretrained_no_processing(..., **cfg.model_from_pretrained_kwargs)
  warnings.warn(


## Single Prompt
Run one prompt through GPT-2, encode residual stream through SAE, see which features fire per token.

In [5]:
# [Cell 4] SAE encode = matmul + ReLU
# Run prompt through GPT-2, grab layer 8 residual stream, encode through SAE

prompt = "The capital of France is"
_, cache = model.run_with_cache(prompt)
x = cache["blocks.8.hook_resid_pre"]  # GPT-2 residual: [1, seq_len, 768]

# SAE encode: features = ReLU((x - b_dec) @ W_enc + b_enc)
features = torch.relu((x - b_dec) @ W_enc + b_enc)  # [1, seq_len, 24576]

print(f"GPT-2 residual shape: {x.shape}")
print(f"SAE features shape:   {features.shape}")

# How many SAE features fire per token?
tokens = model.to_str_tokens(prompt)
for i, tok in enumerate(tokens):
    n_active = (features[0, i] > 0).sum().item()
    print(f"  Token '{tok}': {n_active} active features out of {features.shape[-1]}")

GPT-2 residual shape: torch.Size([1, 6, 768])
SAE features shape:   torch.Size([1, 6, 24576])
  Token '<|endoftext|>': 8 active features out of 24576
  Token 'The': 8 active features out of 24576
  Token ' capital': 35 active features out of 24576
  Token ' of': 66 active features out of 24576
  Token ' France': 47 active features out of 24576
  Token ' is': 51 active features out of 24576


In [6]:
# [Cell 5] Top SAE features for last token
last_acts = features[0, -1]  # [24576]
top = torch.topk(last_acts, 10)

print(f"Top 10 SAE features on last token '{tokens[-1]}':")
print(f"{'Feature ID':>12} | {'Activation':>10}")
print("-" * 27)
for fid, val in zip(top.indices, top.values):
    print(f"{fid.item():>12} | {val.item():>10.3f}")

Top 10 SAE features on last token ' is':
  Feature ID | Activation
---------------------------
       21000 |     10.298
       18220 |      8.410
       14430 |      7.472
        1442 |      6.797
       19805 |      6.314
       21496 |      6.313
        6863 |      6.239
       12013 |      5.085
       19182 |      4.214
       15547 |      3.901


## Steering
Add SAE decoder direction to GPT-2's residual stream to shift output toward a concept.

In [7]:
# [Cell 6] Steering = extract one row from W_dec, add to GPT-2 residual stream
FEATURE_ID = top.indices[0].item()
COEFF = 10.0

# Steering vector = one row of SAE decoder matrix, scaled
steer = COEFF * W_dec[FEATURE_ID]  # [768]

def steering_hook(resid, hook):
    resid[:, :, :] += steer
    return resid

prompt = "I think the best city in the world is"
print("=== Without steering ===")
print(model.generate(prompt, max_new_tokens=50, temperature=0.7))

print(f"\n=== With steering (SAE feature {FEATURE_ID}, coeff={COEFF}) ===")
with model.hooks(fwd_hooks=[("blocks.8.hook_resid_pre", steering_hook)]):
    print(model.generate(prompt, max_new_tokens=50, temperature=0.7))

=== Without steering ===


  0%|          | 0/50 [00:00<?, ?it/s]

I think the best city in the world is LA, because it is so amazing. LA has been growing since I was a kid. I have seen my city grow and I am proud of it. I am very proud of that. If I were to move to LA I'd be one of

=== With steering (SAE feature 21000, coeff=10.0) ===


  0%|          | 0/50 [00:00<?, ?it/s]

I think the best city in the world is probably not New York City. I think it almost looks like Chicago, but I don't think it's actually that big of a deal. Well, it's interesting to look at New York City. It's a major hub for commerce, and you


## Feature Search
Run multiple prompts to find which SAE features correspond to a concept, then verify what a feature represents.

In [ ]:
# [Cell 7] Concept -> feature search
from collections import Counter

CONCEPT = "Golden Gate Bridge"
search_prompts = [
    f"The {CONCEPT} looks cool",
    f"The meaning of {CONCEPT} is",
    f"{CONCEPT} is something that everyone",
    f"The landmark {CONCEPT}",
    f"People often associate {CONCEPT} with",
]

concept_words = [w.lower() for w in CONCEPT.split()]
counts = Counter()
max_acts = {}

for p in search_prompts:
    toks = model.to_str_tokens(p)
    _, c = model.run_with_cache(p)
    x = c["blocks.8.hook_resid_pre"]
    feats = torch.relu((x - b_dec) @ W_enc + b_enc)
    for i, tok in enumerate(toks):
        if any(w in tok.strip().lower() for w in concept_words):
            top10 = torch.topk(feats[0, i], 10)
            for fid, val in zip(top10.indices.tolist(), top10.values.tolist()):
                counts[fid] += 1
                max_acts[fid] = max(max_acts.get(fid, 0), val)

print(f"SAE features for \"{CONCEPT}\":\n")
print(f"{'Feature ID':>10} | {'Times seen':>10} | {'Max activation':>14}")
print("-" * 42)
for fid, cnt in counts.most_common(15):
    print(f"{fid:>10} | {cnt:>10} | {max_acts[fid]:>14.3f}")

SAE features for "Golden Gate Bridge":

Feature ID | Times seen | Max activation
------------------------------------------
      7525 |         10 |         11.706
       735 |          8 |         12.790
     10261 |          5 |         75.465
      7451 |          5 |          6.995
      8198 |          5 |         27.872
     17840 |          5 |         16.332
     23937 |          5 |         10.786
      3808 |          5 |          7.535
       990 |          5 |          6.821
     17608 |          5 |         41.628
     16031 |          5 |          8.430
      5055 |          5 |          7.304
      6863 |          5 |          4.776
      4587 |          4 |         13.108
     21665 |          4 |          6.182


In [17]:
# [Cell 8] Feature -> concept lookup
# Given a feature ID, find what concept it represents
FEATURE_ID = 10261  # try 974 (Paris), 10261 (Golden), 17608 (Bridge), etc.

probe_prompts = [
    "The Golden Gate Bridge spans San Francisco Bay",
    "San Francisco is in California",
    "The Brooklyn Bridge connects Manhattan and Brooklyn",
    "Bridges are built over rivers and bays",
    "The Bay Area has many tech companies",
    "The Golden Gate Bridge is huge",
]

results = []
for p in probe_prompts:
    toks = model.to_str_tokens(p)
    _, c = model.run_with_cache(p)
    x = c["blocks.8.hook_resid_pre"]
    feats = torch.relu((x - b_dec) @ W_enc + b_enc)
    for i, tok in enumerate(toks):
        val = feats[0, i, FEATURE_ID].item()
        if val > 0:
            results.append((val, tok, p))

results.sort(reverse=True)

print(f"Feature {FEATURE_ID} fires on:\n")
for val, tok, prompt in results[:15]:
    print(f"  {val:>8.3f}  '{tok}'  <- {prompt}")

Feature 10261 fires on:

    73.706  ' Golden'  <- The Golden Gate Bridge spans San Francisco Bay


## Multi-Feature Steering
Combine multiple SAE feature directions to steer toward a complex concept.

In [22]:
# [Cell 9] Golden Gate Bridge steering — combine multiple SAE feature directions
gg_features = {
    10261: 40.0,  # "Gate"
    7525: 300
}

# Steering vector = sum of scaled SAE decoder rows
steer = sum(coeff * W_dec[fid] for fid, coeff in gg_features.items())  # [768]

def gg_hook(resid, hook):
    resid[:, :, :] += steer
    return resid

prompt = "I think the best place to visit is"
print("=== Without steering ===")
print(model.generate(prompt, max_new_tokens=50, temperature=0.7))

print(f"\n=== With Golden Gate Bridge steering ===")
with model.hooks(fwd_hooks=[("blocks.8.hook_resid_pre", gg_hook)]):
    print(model.generate(prompt, max_new_tokens=50, temperature=0.7))

=== Without steering ===


  0%|          | 0/50 [00:00<?, ?it/s]

I think the best place to visit is in Africa. I can't live in this country without cycling. I've been cycling for 15 years.

I've been cycling for years and I've loved doing it. I've gone to all the races, I've been at the water

=== With Golden Gate Bridge steering ===


  0%|          | 0/50 [00:00<?, ?it/s]

I think the best place to visit is, City City,,, City-,,,-,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
